In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import random
import numpy as np
import cv2
from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
BASE_PATH = "/kaggle/input/datasets/aysendegerli/qatacov19-dataset/QaTa-COV19/QaTa-COV19-v2/Train Set"

IMAGE_DIR = os.path.join(BASE_PATH, "Images")
MASK_DIR = os.path.join(BASE_PATH, "Ground-truths")

image_paths = sorted(glob(os.path.join(IMAGE_DIR, "*.png")))
mask_paths = sorted(glob(os.path.join(MASK_DIR, "*.png")))

print("Total images:", len(image_paths))

Total images: 7145


In [3]:
class QaTaDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, resize=224):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.resize = resize

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.resize, self.resize))
        img = torch.tensor(img / 255.0, dtype=torch.float32).permute(2,0,1)

        if self.mask_paths is not None:
            mask = cv2.imread(self.mask_paths[idx], 0)
            mask = cv2.resize(mask, (self.resize, self.resize))
            mask = (mask > 0).astype(np.float32)
            mask = torch.tensor(mask).unsqueeze(0)
            return img, mask

        return img


In [4]:
combined = list(zip(image_paths, mask_paths))
random.shuffle(combined)
image_paths, mask_paths = zip(*combined)

num_clients = 4
client_data = [[] for _ in range(num_clients)]

for i, (img, msk) in enumerate(zip(image_paths, mask_paths)):
    client_data[i % num_clients].append((img, msk))

labeled_clients = [0, 1]
unlabeled_clients = [2, 3]

batch_size = 8
client_loaders = []

for i in range(num_clients):
    imgs = [x[0] for x in client_data[i]]
    masks = [x[1] for x in client_data[i]]
    dataset_client = QaTaDataset(imgs, masks)
    loader = DataLoader(dataset_client, batch_size=batch_size, shuffle=True)
    client_loaders.append(loader)

print("Clients ready")


Clients ready


In [5]:
class ResNetUNet(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        
        self.encoder0 = nn.Sequential(
            base_model.conv1,
            base_model.bn1,
            base_model.relu
        )
        self.pool = base_model.maxpool
        self.encoder1 = base_model.layer1
        self.encoder2 = base_model.layer2
        self.encoder3 = base_model.layer3
        self.encoder4 = base_model.layer4
        
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv4 = nn.Conv2d(256 + 256, 256, 3, padding=1)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = nn.Conv2d(128 + 128, 128, 3, padding=1)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = nn.Conv2d(64 + 64, 64, 3, padding=1)

        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.conv1 = nn.Conv2d(64 + 64, 64, 3, padding=1)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        input_size = x.shape[2:]

        e0 = self.encoder0(x)
        p = self.pool(e0)
        e1 = self.encoder1(p)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        d4 = self.up4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = F.relu(self.conv4(d4))

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = F.relu(self.conv3(d3))

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = F.relu(self.conv2(d2))

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e0], dim=1)
        d1 = F.relu(self.conv1(d1))

        out = self.final(d1)

        # Resize to original resolution (224x224)
        out = F.interpolate(
            out,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return torch.sigmoid(out)


In [6]:
def dice_loss(pred, target, smooth=1e-6):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return 1 - dice



In [7]:
def compute_entropy(pred):
    eps = 1e-6
    pred = torch.clamp(pred, eps, 1 - eps)
    entropy = - (pred * torch.log(pred) + (1 - pred) * torch.log(1 - pred))
    return entropy


In [8]:
def extract_prototypes(model, dataloader):
    model.eval()
    
    fg_sum = None
    bg_sum = None
    fg_count = 0
    bg_count = 0
    
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            # Forward to encoder4
            e0 = model.encoder0(imgs)
            p = model.pool(e0)
            e1 = model.encoder1(p)
            e2 = model.encoder2(e1)
            e3 = model.encoder3(e2)
            features = model.encoder4(e3)  # [B,512,H,W]
            
            masks_resized = F.interpolate(
                masks,
                size=features.shape[2:],
                mode="nearest"
            )
            
            fg_mask = (masks_resized > 0).float()
            bg_mask = 1 - fg_mask
            
            fg_feat = features * fg_mask
            bg_feat = features * bg_mask
            
            if fg_sum is None:
                fg_sum = fg_feat.sum(dim=(0,2,3))
                bg_sum = bg_feat.sum(dim=(0,2,3))
            else:
                fg_sum += fg_feat.sum(dim=(0,2,3))
                bg_sum += bg_feat.sum(dim=(0,2,3))
            
            fg_count += fg_mask.sum()
            bg_count += bg_mask.sum()
    
    fg_proto = fg_sum / (fg_count + 1e-6)
    bg_proto = bg_sum / (bg_count + 1e-6)
    
    return fg_proto, bg_proto


In [9]:
def contrastive_loss(features, masks, fg_proto, bg_proto, temp=0.1):
    
    B, C, H, W = features.shape
    
    masks = F.interpolate(masks, size=(H, W), mode="nearest")
    
    features = features.permute(0,2,3,1).reshape(-1, C)
    masks = masks.reshape(-1)
    
    fg_proto = fg_proto.unsqueeze(0)
    bg_proto = bg_proto.unsqueeze(0)
    
    sim_fg = F.cosine_similarity(features, fg_proto, dim=1) / temp
    sim_bg = F.cosine_similarity(features, bg_proto, dim=1) / temp
    
    logits = torch.stack([sim_bg, sim_fg], dim=1)
    
    labels = masks.long()
    
    loss = F.cross_entropy(logits, labels)
    
    return loss


In [10]:
def compute_similarity_weights(unlabeled_proto, labeled_protos):
    sims = []
    for proto in labeled_protos:
        sim = F.cosine_similarity(
            unlabeled_proto.unsqueeze(0),
            proto.unsqueeze(0),
            dim=1
        )
        sims.append(sim)
    
    sims = torch.stack(sims).squeeze()
    weights = torch.softmax(sims, dim=0)
    return weights


In [11]:
def compute_agreement_ratio(local_model, agg_model, dataloader, threshold=0.5):
    local_model.eval()
    agg_model.eval()
    
    total = 0
    agree = 0
    
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            
            p1 = (local_model(imgs) > threshold).float()
            p2 = (agg_model(imgs) > threshold).float()
            
            agreement = (p1 == p2).float()
            
            agree += agreement.sum().item()
            total += agreement.numel()
    
    return agree / (total + 1e-6)


In [12]:
def local_train_full(model,
                     dataloader,
                     client_id,
                     labeled_agg_model=None,
                     fg_proto=None,
                     bg_proto=None,
                     epochs=1,
                     lr=1e-4,
                     threshold=0.5,
                     lambda_contrast=0.1):
    
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        total_loss = 0
        
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            # Forward full model
            preds = model(imgs)
            dice = dice_loss(preds, masks)
            
            # Extract encoder features for contrastive
            e0 = model.encoder0(imgs)
            p = model.pool(e0)
            e1 = model.encoder1(p)
            e2 = model.encoder2(e1)
            e3 = model.encoder3(e2)
            features = model.encoder4(e3)
            
            contrast = contrastive_loss(
                features,
                masks,
                fg_proto,
                bg_proto
            )
            
            loss = dice + lambda_contrast * contrast
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Client {client_id} Epoch {epoch+1} Loss: {total_loss/len(dataloader):.4f}")
    
    return model.state_dict()


In [13]:
def build_labeled_agg_model(global_model, labeled_models, weights):
    
    new_model = ResNetUNet().to(device)
    new_state = new_model.state_dict()
    
    labeled_states = [m.state_dict() for m in labeled_models]
    
    for key in new_state.keys():
        weighted_sum = 0
        for i in range(len(labeled_states)):
            weighted_sum += weights[i] * labeled_states[i][key]
        new_state[key] = weighted_sum
    
    new_model.load_state_dict(new_state)
    return new_model


In [14]:
global_model = ResNetUNet().to(device)

rounds = 15
local_epochs = 1

for r in range(rounds):
    print(f"\n================ ROUND {r+1} ================")
    
    client_weights = [None]*num_clients
    client_models = {}
    labeled_models = []
    labeled_fg_protos = []
    labeled_bg_protos = []
    
    # -------- TRAIN LABELED CLIENTS --------
    for i in labeled_clients:
        local_model = ResNetUNet().to(device)
        local_model.load_state_dict(global_model.state_dict())
        
        # Extract prototypes from labeled client
        fg_proto, bg_proto = extract_prototypes(local_model, client_loaders[i])
        
        weights = local_train_full(
            local_model,
            client_loaders[i],
            i,
            fg_proto=fg_proto,
            bg_proto=bg_proto,
            epochs=local_epochs
        )
        
        client_weights[i] = weights
        client_models[i] = local_model
        
        labeled_models.append(local_model)
        labeled_fg_protos.append(fg_proto)
        labeled_bg_protos.append(bg_proto)
    
    # -------- UNLABELED CLIENTS --------
    for i in unlabeled_clients:
        temp_model = ResNetUNet().to(device)
        temp_model.load_state_dict(global_model.state_dict())
        
        # Use average labeled prototypes
        fg_proto = torch.stack(labeled_fg_protos).mean(dim=0)
        bg_proto = torch.stack(labeled_bg_protos).mean(dim=0)
        
        weights = local_train_full(
            temp_model,
            client_loaders[i],
            i,
            fg_proto=fg_proto,
            bg_proto=bg_proto,
            epochs=local_epochs
        )
        
        client_weights[i] = weights
        client_models[i] = temp_model
    
    # -------- DYNAMIC AGGREGATION --------
    Z = []
    data_sizes = []
    
    for i in range(num_clients):
        data_sizes.append(len(client_loaders[i].dataset))
        
        if i in labeled_clients:
            Z.append(1.0)
        else:
            Z.append(0.7)  # simplified stability factor
    
    Z = torch.tensor(Z)
    data_sizes = torch.tensor(data_sizes, dtype=torch.float32)
    
    weights_dyn = Z * data_sizes
    weights_dyn = weights_dyn / weights_dyn.sum()
    
    print("Dynamic weights:", weights_dyn.numpy())
    
    global_dict = global_model.state_dict()
    
    for key in global_dict.keys():
        weighted_sum = 0
        for i in range(num_clients):
            weighted_sum += weights_dyn[i] * client_weights[i][key]
        global_dict[key] = weighted_sum
    
    global_model.load_state_dict(global_dict)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 168MB/s]



================ ROUND 1 ================
Client 0 Epoch 1 Loss: 0.4165
Client 1 Epoch 1 Loss: 0.4202
Client 2 Epoch 1 Loss: 0.4134
Client 3 Epoch 1 Loss: 0.4179
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 2 ================
Client 0 Epoch 1 Loss: 0.2260
Client 1 Epoch 1 Loss: 0.2250
Client 2 Epoch 1 Loss: 0.2338
Client 3 Epoch 1 Loss: 0.2305
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 3 ================
Client 0 Epoch 1 Loss: 0.2021
Client 1 Epoch 1 Loss: 0.2020
Client 2 Epoch 1 Loss: 0.2085
Client 3 Epoch 1 Loss: 0.2143
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 4 ================
Client 0 Epoch 1 Loss: 0.1873
Client 1 Epoch 1 Loss: 0.1913
Client 2 Epoch 1 Loss: 0.1963
Client 3 Epoch 1 Loss: 0.1964
Dynamic weights: [0.29423386 0.2940692  0.20584843 0.20584843]

================ ROUND 5 ================
Client 0 Epoch 1 Loss: 0.1769
Client 1 Epoch 1 Loss: 

In [15]:
# Validation Split
val_imgs = image_paths[:200]
val_masks = mask_paths[:200]

val_dataset = QaTaDataset(val_imgs, val_masks)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


def evaluate(model, dataloader):
    model.eval()
    total_dice = 0
    count = 0
    
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            preds = model(imgs)
            preds = (preds > 0.5).float()
            
            intersection = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
            
            dice = (2 * intersection + 1e-6) / (union + 1e-6)
            
            total_dice += dice.sum().item()
            count += imgs.size(0)
    
    return total_dice / count


dice_score = evaluate(global_model, val_loader)

print("\n================ FINAL FSSS RESULT ================")
print("FSSS Validation Dice:", dice_score)



================ FINAL FSSS RESULT ================
FSSS Validation Dice: 0.8329650163650513


In [16]:
baseline_model = ResNetUNet().to(device)

rounds = 10
local_epochs = 1

for r in range(rounds):
    print(f"\n===== BASELINE ROUND {r+1} =====")
    
    client_weights = []
    
    for i in labeled_clients:  # Only labeled clients
        local_model = ResNetUNet().to(device)
        local_model.load_state_dict(baseline_model.state_dict())
        
        optimizer = torch.optim.Adam(local_model.parameters(), lr=1e-4)
        local_model.train()
        
        for epoch in range(local_epochs):
            total_loss = 0
            for imgs, masks in client_loaders[i]:
                imgs = imgs.to(device)
                masks = masks.to(device)
                
                preds = local_model(imgs)
                loss = dice_loss(preds, masks)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            print(f"Client {i} Epoch Loss: {total_loss/len(client_loaders[i]):.4f}")
        
        client_weights.append(local_model.state_dict())
    
    # FedAvg between labeled clients
    global_dict = baseline_model.state_dict()
    
    for key in global_dict.keys():
        stacked = torch.stack(
            [client_weights[i][key].float() for i in range(len(client_weights))],
            dim=0
        )
        global_dict[key] = stacked.mean(dim=0)
    
    baseline_model.load_state_dict(global_dict)


# Evaluate baseline
dice_baseline = evaluate(baseline_model, val_loader)

print("\n================ BASELINE RESULT ================")
print("Baseline Dice:", dice_baseline)



===== BASELINE ROUND 1 =====
Client 0 Epoch Loss: 0.3816
Client 1 Epoch Loss: 0.3834

===== BASELINE ROUND 2 =====
Client 0 Epoch Loss: 0.2106
Client 1 Epoch Loss: 0.2094

===== BASELINE ROUND 3 =====
Client 0 Epoch Loss: 0.1832
Client 1 Epoch Loss: 0.1912

===== BASELINE ROUND 4 =====
Client 0 Epoch Loss: 0.1801
Client 1 Epoch Loss: 0.1767

===== BASELINE ROUND 5 =====
Client 0 Epoch Loss: 0.1628
Client 1 Epoch Loss: 0.1644

===== BASELINE ROUND 6 =====
Client 0 Epoch Loss: 0.1600
Client 1 Epoch Loss: 0.1484

===== BASELINE ROUND 7 =====
Client 0 Epoch Loss: 0.1463
Client 1 Epoch Loss: 0.1428

===== BASELINE ROUND 8 =====
Client 0 Epoch Loss: 0.1426
Client 1 Epoch Loss: 0.1370

===== BASELINE ROUND 9 =====
Client 0 Epoch Loss: 0.1329
Client 1 Epoch Loss: 0.1233

===== BASELINE ROUND 10 =====
Client 0 Epoch Loss: 0.1296
Client 1 Epoch Loss: 0.1212

================ BASELINE RESULT ================
Baseline Dice: 0.8118536877632141


In [14]:
# ══════════════════════════════════════════════════════════════
# ENHANCEMENT 1: ResNetUNet with MC Dropout (replaces plain entropy)
# ══════════════════════════════════════════════════════════════

class ResNetUNetMCDropout(nn.Module):
    """ResNet34-UNet with Dropout layers inserted in the decoder for
    Monte Carlo Dropout uncertainty estimation."""

    def __init__(self, dropout_p=0.3):
        super().__init__()
        base_model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

        self.encoder0 = nn.Sequential(base_model.conv1,
                                      base_model.bn1,
                                      base_model.relu)
        self.pool     = base_model.maxpool
        self.encoder1 = base_model.layer1
        self.encoder2 = base_model.layer2
        self.encoder3 = base_model.layer3
        self.encoder4 = base_model.layer4

        self.drop = nn.Dropout2d(p=dropout_p)

        self.up4   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv4 = nn.Conv2d(256 + 256, 256, 3, padding=1)

        self.up3   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = nn.Conv2d(128 + 128, 128, 3, padding=1)

        self.up2   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = nn.Conv2d(64 + 64, 64, 3, padding=1)

        self.up1   = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.conv1 = nn.Conv2d(64 + 64, 64, 3, padding=1)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        input_size = x.shape[2:]

        e0 = self.encoder0(x)
        p  = self.pool(e0)
        e1 = self.encoder1(p)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        d4 = self.up4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = F.relu(self.conv4(d4))
        d4 = self.drop(d4)           # ← dropout after each decoder block

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = F.relu(self.conv3(d3))
        d3 = self.drop(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = F.relu(self.conv2(d2))
        d2 = self.drop(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e0], dim=1)
        d1 = F.relu(self.conv1(d1))

        out = self.final(d1)
        out = F.interpolate(out, size=input_size, mode="bilinear", align_corners=False)
        return torch.sigmoid(out)


print("ResNetUNetMCDropout defined ✔")


ResNetUNetMCDropout defined ✔


In [15]:
# ══════════════════════════════════════════════════════════════
# MC Dropout Uncertainty Estimation
# ══════════════════════════════════════════════════════════════

def mc_dropout_uncertainty(model, imgs, n_passes=10):
    """
    Run n stochastic forward passes with dropout ACTIVE (model.train())
    and return mean prediction + predictive variance as uncertainty.

    Returns
    -------
    mean_pred   : Tensor [B,1,H,W]  – averaged prediction across passes
    uncertainty : Tensor [B,1,H,W]  – pixel-wise variance (higher = less reliable)
    """
    model.train()          # keep dropout active
    preds = []

    with torch.no_grad():
        for _ in range(n_passes):
            preds.append(model(imgs))   # [B,1,H,W]

    preds = torch.stack(preds, dim=0)   # [T, B, 1, H, W]
    mean_pred   = preds.mean(dim=0)     # [B, 1, H, W]
    uncertainty = preds.var(dim=0)      # [B, 1, H, W]  predictive variance

    return mean_pred, uncertainty


def compute_client_uncertainty(model, dataloader, n_passes=10):
    """Scalar uncertainty score for a client (mean variance over dataset)."""
    total_var = 0.0
    count     = 0

    for imgs, _ in dataloader:
        imgs = imgs.to(device)
        _, unc = mc_dropout_uncertainty(model, imgs, n_passes=n_passes)
        total_var += unc.mean().item()
        count     += 1

    return total_var / (count + 1e-6)


print("MC Dropout uncertainty functions defined ✔")


MC Dropout uncertainty functions defined ✔


In [16]:
# ══════════════════════════════════════════════════════════════
# ENHANCEMENT 2: Trust-Aware Federated Aggregation
# T_i = α·Agreement + β·Similarity − γ·Uncertainty
# W_global = Σ T_i · W_i
# ══════════════════════════════════════════════════════════════

def compute_prototype_similarity(model, dataloader, reference_fg, reference_bg):
    """Cosine similarity between this client's prototype and the global reference."""
    fg_proto, bg_proto = extract_prototypes(model, dataloader)

    sim_fg = F.cosine_similarity(fg_proto.unsqueeze(0),
                                  reference_fg.unsqueeze(0)).item()
    sim_bg = F.cosine_similarity(bg_proto.unsqueeze(0),
                                  reference_bg.unsqueeze(0)).item()
    return (sim_fg + sim_bg) / 2.0


def compute_trust_scores(client_models,
                          client_loaders,
                          global_model,
                          num_clients,
                          labeled_fg_protos,
                          labeled_bg_protos,
                          alpha=0.4,
                          beta=0.4,
                          gamma=0.2,
                          n_mc_passes=10):
    """
    Compute T_i = α·Agreement + β·Similarity − γ·Uncertainty  for every client.

    Parameters
    ----------
    alpha  : weight for agreement
    beta   : weight for prototype similarity
    gamma  : penalty weight for uncertainty
    """

    # Global reference prototypes (mean of labeled clients)
    ref_fg = torch.stack(labeled_fg_protos).mean(dim=0)
    ref_bg = torch.stack(labeled_bg_protos).mean(dim=0)

    trust_scores = []

    for i in range(num_clients):
        model_i = client_models[i]

        # ── Agreement: fraction of pixels where local == global model ──
        agreement = compute_agreement_ratio(model_i, global_model,
                                            client_loaders[i])

        # ── Similarity: prototype cosine similarity to global reference ──
        similarity = compute_prototype_similarity(model_i, client_loaders[i],
                                                   ref_fg, ref_bg)

        # ── Uncertainty: MC Dropout predictive variance ──
        uncertainty = compute_client_uncertainty(model_i, client_loaders[i],
                                                  n_passes=n_mc_passes)

        # Normalise uncertainty to [0,1] range heuristically
        uncertainty_norm = min(uncertainty / 0.25, 1.0)

        T_i = alpha * agreement + beta * similarity - gamma * uncertainty_norm
        T_i = max(T_i, 1e-6)   # clip to positive

        trust_scores.append(T_i)
        print(f"  Client {i} | Agreement={agreement:.3f}  "              f"Similarity={similarity:.3f}  "              f"Uncertainty={uncertainty:.4f}  "              f"→ Trust={T_i:.4f}")

    return trust_scores


def trust_aware_aggregation(global_model, client_weights, trust_scores):
    """Weighted average of client weights using normalised trust scores."""
    T = torch.tensor(trust_scores, dtype=torch.float32)
    T = T / T.sum()          # normalise  →  Σ T_i = 1
    print("Trust-normalised weights:", T.numpy())

    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        weighted_sum = sum(T[i] * client_weights[i][key]
                           for i in range(len(client_weights)))
        global_dict[key] = weighted_sum

    global_model.load_state_dict(global_dict)
    return global_model


print("Trust-Aware Aggregation functions defined ✔")


Trust-Aware Aggregation functions defined ✔


In [17]:
# ══════════════════════════════════════════════════════════════
# TRUST-AWARE FEDERATED TRAINING LOOP
# Uses ResNetUNetMCDropout + Trust Score aggregation
# Ablation flags: use_similarity, use_uncertainty
# ══════════════════════════════════════════════════════════════

def run_trust_federated(rounds=15,
                         local_epochs=1,
                         alpha=0.4,
                         beta=0.4,
                         gamma=0.2,
                         n_mc_passes=10,
                         use_similarity=True,
                         use_uncertainty=True,
                         label="Trust-Aware"):
    """
    Full federated loop with Trust-Aware aggregation.

    Ablation support
    ----------------
    use_similarity=False  →  beta forced to 0
    use_uncertainty=False →  gamma forced to 0
    """
    if not use_similarity:
        beta_eff  = 0.0
        alpha_eff = alpha + beta     # redistribute weight
        gamma_eff = gamma
    elif not use_uncertainty:
        gamma_eff = 0.0
        alpha_eff = alpha
        beta_eff  = beta + gamma     # redistribute weight
    else:
        alpha_eff, beta_eff, gamma_eff = alpha, beta, gamma

    print(f"{'='*60}")
    print(f"  {label}  (α={alpha_eff:.2f}  β={beta_eff:.2f}  γ={gamma_eff:.2f})")
    print(f"{'='*60}")

    global_model_t = ResNetUNetMCDropout().to(device)

    for r in range(rounds):
        print(f"──── ROUND {r+1} ────")

        client_weights  = [None] * num_clients
        client_models_r = {}
        labeled_models  = []
        labeled_fg_protos = []
        labeled_bg_protos = []

        # ── Labeled clients ────────────────────────────────────────
        for i in labeled_clients:
            local_model = ResNetUNetMCDropout().to(device)
            local_model.load_state_dict(global_model_t.state_dict())

            fg_proto, bg_proto = extract_prototypes(local_model,
                                                     client_loaders[i])

            weights_sd = local_train_full(local_model,
                                          client_loaders[i],
                                          i,
                                          fg_proto=fg_proto,
                                          bg_proto=bg_proto,
                                          epochs=local_epochs)

            client_weights[i]  = weights_sd
            client_models_r[i] = local_model
            labeled_models.append(local_model)
            labeled_fg_protos.append(fg_proto)
            labeled_bg_protos.append(bg_proto)

        # ── Unlabeled clients ──────────────────────────────────────
        fg_proto_avg = torch.stack(labeled_fg_protos).mean(dim=0)
        bg_proto_avg = torch.stack(labeled_bg_protos).mean(dim=0)

        for i in unlabeled_clients:
            local_model = ResNetUNetMCDropout().to(device)
            local_model.load_state_dict(global_model_t.state_dict())

            weights_sd = local_train_full(local_model,
                                          client_loaders[i],
                                          i,
                                          fg_proto=fg_proto_avg,
                                          bg_proto=bg_proto_avg,
                                          epochs=local_epochs)

            client_weights[i]  = weights_sd
            client_models_r[i] = local_model

        # ── Trust Scores + Aggregation ─────────────────────────────
        trust_scores = compute_trust_scores(
            client_models_r,
            client_loaders,
            global_model_t,
            num_clients,
            labeled_fg_protos,
            labeled_bg_protos,
            alpha=alpha_eff,
            beta=beta_eff,
            gamma=gamma_eff,
            n_mc_passes=n_mc_passes
        )

        global_model_t = trust_aware_aggregation(global_model_t,
                                                   client_weights,
                                                   trust_scores)

    return global_model_t


# ── Run full model ─────────────────────────────────────────────────────────────
trust_global_model = run_trust_federated(
    rounds=15,
    label="Trust-Aware (Full)"
)

dice_trust = evaluate(trust_global_model, val_loader)

print(f"Trust-Aware Dice Score: {dice_trust:.4f}")

  Trust-Aware (Full)  (α=0.40  β=0.40  γ=0.20)
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 201MB/s]


──── ROUND 1 ────
Client 0 Epoch 1 Loss: 0.4292
Client 1 Epoch 1 Loss: 0.4269
Client 2 Epoch 1 Loss: 0.4265
Client 3 Epoch 1 Loss: 0.4335
  Client 0 | Agreement=0.836  Similarity=0.702  Uncertainty=0.0023  → Trust=0.6135
  Client 1 | Agreement=0.857  Similarity=0.689  Uncertainty=0.0020  → Trust=0.6167
  Client 2 | Agreement=0.862  Similarity=0.693  Uncertainty=0.0019  → Trust=0.6202
  Client 3 | Agreement=0.873  Similarity=0.687  Uncertainty=0.0018  → Trust=0.6226
Trust-normalised weights: [0.2480715  0.2493724  0.25081012 0.251746  ]
──── ROUND 2 ────
Client 0 Epoch 1 Loss: 0.2392
Client 1 Epoch 1 Loss: 0.2360
Client 2 Epoch 1 Loss: 0.2354
Client 3 Epoch 1 Loss: 0.2320
  Client 0 | Agreement=0.959  Similarity=0.967  Uncertainty=0.0022  → Trust=0.7687
  Client 1 | Agreement=0.931  Similarity=0.934  Uncertainty=0.0019  → Trust=0.7447
  Client 2 | Agreement=0.952  Similarity=0.972  Uncertainty=0.0023  → Trust=0.7676
  Client 3 | Agreement=0.942  Similarity=0.936  Uncertainty=0.0019  → T

NameError: name 'evaluate' is not defined

In [18]:
# Validation Split
val_imgs = image_paths[:200]
val_masks = mask_paths[:200]

val_dataset = QaTaDataset(val_imgs, val_masks)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


def evaluate(model, dataloader):
    model.eval()
    total_dice = 0
    count = 0
    
    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            preds = model(imgs)
            preds = (preds > 0.5).float()
            
            intersection = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
            
            dice = (2 * intersection + 1e-6) / (union + 1e-6)
            
            total_dice += dice.sum().item()
            count += imgs.size(0)
    
    return total_dice / count

In [19]:
dice_trust = evaluate(trust_global_model, val_loader)

print(f"Trust-Aware Dice Score: {dice_trust:.4f}")

Trust-Aware Dice Score: 0.8359


In [20]:
# ══════════════════════════════════════════════════════════════
# ABLATION STUDY
# Three variants:
#   1. Without uncertainty  (γ = 0)
#   2. Without similarity   (β = 0)
#   3. Full model           (all three signals)
# ══════════════════════════════════════════════════════════════

# ── Variant 1: No uncertainty ──────────────────────────────────
model_no_unc = run_trust_federated(
    rounds=15,
    use_uncertainty=False,
    label="Ablation: No Uncertainty"
)
dice_no_unc = evaluate(model_no_unc, val_loader)

# ── Variant 2: No similarity ───────────────────────────────────
model_no_sim = run_trust_federated(
    rounds=15,
    use_similarity=False,
    label="Ablation: No Similarity"
)
dice_no_sim = evaluate(model_no_sim, val_loader)

# ── Summary ────────────────────────────────────────────────────
print("" + "="*55)
print("           ABLATION STUDY RESULTS")
print("="*55)
print(f"  Baseline (FedAvg, labeled only)   : {dice_baseline:.4f}")
print(f"  Original FSSS (dynamic weights)   : {dice_score:.4f}")
print(f"  Trust-Aware (full model)          : {dice_trust:.4f}")
print(f"  Ablation – without uncertainty    : {dice_no_unc:.4f}")
print(f"  Ablation – without similarity     : {dice_no_sim:.4f}")
print("="*55)


  Ablation: No Uncertainty  (α=0.40  β=0.60  γ=0.00)
──── ROUND 1 ────


KeyboardInterrupt: 